# OghmaNano Fabry–Pérot FDTD → TMM 对齐验证

Oghma v8.1 FDTD 项目 [`01_hello_fabry_perot`](../../../database/og/oghma_projects/fabry_perot/01_hello_fabry_perot)：
用 [`00_software_alignment_skills.md`](00_software_alignment_skills.md) 正向构造 `I_new(λ)` 驱动 **emission TMM**（hs 偶极子 @ `z_source`，`u=0`）与 **被动 TMM**，
经 [`oghma_fdtd_alignment.py`](oghma_fdtd_alignment.py) 统一数据读取与 `evaluate_case` 比对。

参考 [`05_bragg_grating.ipynb`](05_bragg_grating.ipynb) Case A–D 流程（Fabry 无 Bragg 阻带掩膜）。

**运行前提**：在 `simulation_core` 根目录执行 `source scripts/init-simulation-build-env.sh build`，再 `./assets/ipynb/simulation/TMM/run_tmm.sh jupyter`（或在已 source 的环境中打开本 notebook）。须具备 `SIMULATION_ARTIFACTS_DIR`（Release `build/`）与 `SIMULATION_DATABASE_DIR`（YAML `assets/database`）；**勿**使用 `init-toykits-build-env.sh` / `.simulation_toolkits`。


In [ ]:
import numpy as np

from oghma_runtime import bootstrap_tmm_session

_, RUNTIME, TMM_DIR = bootstrap_tmm_session()
import simulation

from oghma_core import _structure_pos_from_layers, fdtd_input_combine_power
from oghma_fabry import (
    build_fabry_perot_stack,
    compute_fabry_emission_case_bc_spectra,
    compute_fabry_emission_transfer,
    compute_fabry_lam_e_norm,
    compute_fabry_passive_transfer,
    compute_fabry_reflection_ahead_of_z,
    fabry_perot_interface_R_at_wl,
    fabry_perot_stack_labels,
    fabry_perot_thicknesses_um,
    parse_fabry_perot_geometry,
    plot_fabry_perot_fdtd_stack_section,
)
from oghma_fdtd_alignment import (
    display_alignment_reports,
    evaluate_case,
    load_fdtd_alignment_bundle,
    plot_case_summary_grid,
    plot_forward_source_diagnostic,
    s_in_times_intensity,
)
from oghma_fdtd_source import integrate_incoherent_power, predict_fdtd_output_spectrum_incoherent

bundle = load_fdtd_alignment_bundle(
    "fabry_perot", "01_hello_fabry_perot",
    parse_geometry_fn=parse_fabry_perot_geometry,
)
MASK_FULL = (bundle.wl_um >= 0.4) & (bundle.wl_um <= 0.6)
MASK_RES = (bundle.wl_um >= 0.48) & (bundle.wl_um <= 0.52)

print(f"RUNTIME={RUNTIME}")
print(f"cavity L={bundle.geom.cavity_length_um * 1e3} nm, mirror={bundle.geom.mirror_thickness_um * 1e3} nm")
print(f"λ₁=2nL≈{bundle.geom.fundamental_resonance_um * 1e3:.0f} nm")
print(f"z_source={bundle.layout.z_source_um * 1e3:.0f} nm, z_det_in={bundle.layout.z_detector_in_um * 1e3:.0f} nm")
print(f"λ: {bundle.wl_um[0]*1e3:.0f}–{bundle.wl_um[-1]*1e3:.0f} nm ({len(bundle.wl_um)} pts)")
print(f"FDTD source: {bundle.fdtd_src.waveform}, Ey={bundle.fdtd_src.excite_ey}")
print(f"emission dipole: orient=hs, u=0, z={bundle.layout.z_source_um * 1e3:.0f} nm")


## §1 器件堆栈

物理 FDTD 膜系；两侧半无限 air 对应 PML。


In [ ]:
layers_demo = build_fabry_perot_stack(
    bundle.geom, 500.0, simulation
)
pos = _structure_pos_from_layers(layers_demo)
labels = fabry_perot_stack_labels(bundle.geom)
thicknesses_nm = fabry_perot_thicknesses_um(bundle.geom)
print("TMM 界面 z (nm):", np.round(pos, 2))
print(f"有限层总厚: {bundle.geom.total_film_thickness_um * 1e3} nm")
R500 = fabry_perot_interface_R_at_wl(
    bundle.geom, 500.0, simulation
)
print(f"TMM R @ 500 nm (air|mirror slab): {R500:.3f}")
plot_fabry_perot_fdtd_stack_section(
    bundle.geom,
    bundle.layout,
    simulation,
)


## §2 等效光源（消除时间项）

预期：`S_in ≈ I_new × G_in`；全部经 `evaluate_case` 比对。


In [ ]:
r_det = compute_fabry_reflection_ahead_of_z(
    bundle.layout.z_detector_in_um * 1e3,
    bundle.geom,
    bundle.layout,
    bundle.wl_um,
    simulation,
    )
g_in = fdtd_input_combine_power(r_det)
s_pred_det0 = predict_fdtd_output_spectrum_incoherent(bundle.i_new, g_in)
s_in_norm = s_in_times_intensity(bundle)

evaluate_case(
    bundle,
    "§2 I_new vs S_IN×Intensity",
    bundle.i_new,
    s_in_norm,
    mask=MASK_FULL,
    mask_label="400,600",
    ylabel="|E|²",
    title="§2: I_new vs S_IN×Intensity",
    tmm_label="I_new",
    baseline_name="S_IN×Int",
    use_stop_band=False,
)
evaluate_case(
    bundle,
    "§2 I_new×G_in vs S_IN",
    s_pred_det0,
    bundle.s_in,
    mask=MASK_FULL,
    mask_label="400,600",
    ylabel="|E|²",
    title="§2: I_new×G_in vs detector0",
    tmm_label="I_new×G_in(ignore temporal coherence)",
    baseline_name="S_IN (det0)",
    use_stop_band=False,
)
# plot_forward_source_diagnostic(bundle, g_in, s_pred_det0, s_in_norm=s_in_norm)


## §3 Case A — 传递比 T(λ)

$T/|1+r|^2$：emission TMM vs 被动 TMM。


In [ ]:
em = compute_fabry_emission_transfer(
    bundle.geom,
    bundle.layout,
    bundle.wl_um,
    simulation,
    z_um=bundle.layout.z_source_um,
    u=0.0,
    orient="hs",
)
t_em = em["t_trans_ratio_fdtd"]
t_passive = compute_fabry_lam_e_norm(
    bundle.geom,
    bundle.layout,
    bundle.wl_um,
    simulation,
    )

evaluate_case(
    bundle,
    "Case A T_em",
    t_em,
    t_passive,
    ylabel="T/G_in",
    title="Case A: emission T/G_in vs passive T/G_in",
    tmm_label="T(emission)/|1+r|^2",
    baseline_name="T/|1+r|^2",
)
# evaluate_case(
#     bundle,
#     "Case A [480,520]",
#     t_em[MASK_RES],
#     t_passive[MASK_RES],
#     bundle.wl_um[MASK_RES],
#     ylabel="T/G_in",
#     title="Case A resonance window",
#     tmm_label="T(emission)/|1+r|^2",
#     baseline_name="T/|1+r|^2",
# )


## §4 Case B / C / D — 出射监视器能量 vs `detector1/lam_E`

- **Case B**：相干 `flux(E_bot)`，源谱 `I_new`
- **Case C**：`T_passive·I_new`（被动非相干）
- **Case D**：Case B vs Case C（emission vs 被动）


In [ ]:
t_passive_t = compute_fabry_passive_transfer(
    bundle.geom,
    bundle.layout,
    bundle.wl_um,
    simulation,
    )
case_bc = compute_fabry_emission_case_bc_spectra(
    simulation,
    bundle.geom,
    bundle.layout,
    bundle.wl_um,
    bundle.s_in,
    spectrum=bundle.i_new,
    t_passive=t_passive_t,
)
s_pred_b = case_bc["s_pred_b"]
s_pred_c = case_bc["s_pred_c"]

evaluate_case(
    bundle,
    "Case B P_bot",
    s_pred_b,
    bundle.s_out,
    ylabel="|E|²",
    title="Case B: coherent flux(E_bot) vs S_OUT",
    tmm_label="flux(E_bot)",
    baseline_name="S_OUT",
)
evaluate_case(
    bundle,
    "Case C T_passive·I_new",
    s_pred_c,
    bundle.s_out,
    ylabel="|E|²",
    title="Case C: T_passive·I_new vs S_OUT",
    tmm_label="T_passive·I_new",
    baseline_name="S_OUT",
)
evaluate_case(
    bundle,
    "Case D emission vs passive",
    s_pred_b,
    s_pred_c,
    ylabel="|E|²",
    title="Case D: emission vs passive",
    tmm_label="Case B",
    baseline_name="Case C",
)

print(f"W_B = ∫ Case B dλ = {integrate_incoherent_power(bundle.wl_um, s_pred_b):.4e}")
print(f"W_C = ∫ Case C dλ = {integrate_incoherent_power(bundle.wl_um, s_pred_c):.4e}")
print(f"W_FDTD = ∫ detector1 lam_E dλ = {integrate_incoherent_power(bundle.wl_um, bundle.s_out):.4e}")


## §5 Case 汇总


In [ ]:
plot_case_summary_grid(
    bundle,
    [
        {
            "title": "Case A",
            "pred": t_em,
            "baseline": t_passive,
            "pred_label": "T_em",
            "baseline_label": "T_passive/G_in",
            "ylabel": "T/G_in",
        },
        {
            "title": "Case B",
            "pred": s_pred_b,
            "baseline": bundle.s_out,
            "pred_label": "Case B",
            "baseline_label": "FDTD S_out",
        },
        {
            "title": "Case C",
            "pred": s_pred_c,
            "baseline": bundle.s_out,
            "pred_label": "Case C",
            "baseline_label": "FDTD S_out",
        },
        {
            "title": "Case D",
            "pred": s_pred_b,
            "baseline": s_pred_c,
            "pred_label": "Case B (emission)",
            "baseline_label": "Case C (passive)",
        },
    ],
)
display_alignment_reports(bundle)
